Key Information Included in Cell 1:

Architecture & Cycle Flow (Variant 1b Specific):

Charge Cycle (4 h): Ambient Air (
𝑇
0
,
𝑃
0
T
0
	​

,P
0
	​

) 
→
→ Single-stage Compressor (
𝑃
0
→
𝑃
m
a
x
P
0
	​

→P
max
	​

) 
→
→ After-cooler (compression heat rejected to cooling water, unrecovered) 
→
→ Constant-pressure reservoir (
16
 bar
16 bar, variable volume).
Discharge Cycle (4 h): Stored air (
16
 bar
,
𝑇
0
16 bar,T
0
	​

) 
→
→ Combustor (fuel heat addition to 
T
I
T
=
550
∘
C
TIT=550
∘
C) 
→
→ Single-stage Turbine (
16
→
1.013
 bar
16→1.013 bar) 
→
→ Exhaust.
Quasi-steady operation: Because the reservoir is maintained at a fixed pressure (
𝑃
m
a
x
=
16
 bar
P
max
	​

=16 bar, variable volume via hydraulic compensation), both charge and discharge operate in steady state (no sliding pressure like in Variant 1a).

Reference Conditions and Target Specifications:

Working fluid: Air, modeled with real-gas properties via CoolProp ("Air")
Ambient temperature 
𝑇
0
T
0
	​

: 
15
∘
C
15
∘
C (
288.15
 K
288.15 K)
Ambient pressure 
𝑃
0
P
0
	​

: 
1.013
 bar
1.013 bar (
101
325
 Pa
101325 Pa)
Storage pressure 
𝑃
m
a
x
P
max
	​

: 
16.0
 bar
16.0 bar (
1.6
×
10
6
 Pa
1.6×10
6
 Pa, constant)
Discharge power: 
1
 MW
1 MW net at generator shaft
Discharge duration: 
4
 h
4 h 
⟹
4
 MWh
⟹4 MWh delivered per cycle
Charge duration: 
4
 h
4 h 
⟹
⟹ sets the compressor mass flow rate

Fixed Component Parameters:

Compressor isentropic efficiency 
𝜂
i
s
,
c
=
0.85
η
is,c
	​

=0.85 (single stage)
Turbine isentropic efficiency 
𝜂
i
s
,
t
=
0.88
η
is,t
	​

=0.88 (single stage)
Mechanical & generator efficiency 
𝜂
m
e
c
h
=
0.98
η
mech
	​

=0.98
Cooler pinch (after-cooler) 
Δ
𝑇
p
i
n
c
h
=
10
 K
ΔT
pinch
	​

=10 K (air cooled to 
𝑇
0
+
10
 K
=
25
∘
C
T
0
	​

+10 K=25
∘
C before storage)
Turbine inlet temperature (
T
I
T
TIT): 
550
∘
C
550
∘
C (
823.15
 K
823.15 K)
Fuel-to-electricity reference efficiency 
𝜂
𝑓
=
0.40
η
f
	​

=0.40

Modelling Assumptions & Physics Constraints:

Reservoir: 0D uniform store. Constant pressure at 
16
 bar
16 bar. Due to slow 4 h discharge, stored air returns to 
𝑇
0
=
15
∘
C
T
0
	​

=15
∘
C prior to discharge and stays near 
𝑇
0
T
0
	​

. Turbine receives 
16
 bar
16 bar air directly with no throttling (preserving exergy).
Heat Rejection: Compression heat rejected to ambient cooling water; no thermal storage (TES). Heat exchanger pressure drop neglected (
Δ
𝑝
=
0
Δp=0).
Combustor: Modeled as simple thermal heat addition 
𝑄
˙
c
o
m
b
u
s
t
i
o
n
=
𝑚
˙
d
i
s
(
ℎ
T
I
T
−
ℎ
s
t
o
r
e
)
Q
˙
	​

combustion
	​

=
m
˙
dis
	​

(h
TIT
	​

−h
store
	​

) (no combustion chemistry).

Performance Metrics & Required Deliverables:

Round-trip efficiency: 
𝜂
=
𝑊
t
u
r
b
i
n
e
𝑊
c
o
m
p
r
e
s
s
o
r
+
𝜂
𝑓
𝑄
c
o
m
b
u
s
t
i
o
n
,
𝜂
𝑓
=
0.40
η=
W
compressor
	​

+η
f
	​

Q
combustion
	​

W
turbine
	​

	​

,η
f
	​

=0.40
Deliverables: Required storage volume 
Δ
𝑉
s
t
o
r
e
ΔV
store
	​

 for 4 MWh, cooler sizing (
𝐴
,
𝑈
𝐴
,
𝑄
˙
c
o
o
l
e
r
,
L
M
T
D
A,UA,
Q
˙
	​

cooler
	​

,LMTD), peak temperatures (
𝑇
c
o
m
p
,
o
u
t
,
T
I
T
T
comp,out
	​

,TIT), electrical/fuel energy input split, and component energy/exergy balances.

Here are some sources i used to return to later (soso)

Return to Gas Turbines for compressor eq and details

Paper on which sile will the pinch temp be occuring based on the Cc and Ch (LMTD) method. https://sci-hub.fr/10.1016/B978-008042839-0/50008-2

Also return to EENG3 S6 Lec 4 for counter flow details and eq

In [ ]:
# --- imports ---
from pathlib import Path
import sys

for root in (Path.cwd(), *Path.cwd().parents):
    for package_root in (root, root / "caldis-0.3.0"):
        if (package_root / "caldis" / "__init__.py").is_file():
            sys.path.insert(0, str(package_root))
            break
    else:
        continue
    break

import numpy as np
import caldis
from CoolProp.CoolProp import PropsSI
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.fluids import CoolPropBackend
from caldis.components import SizingHX
from caldis.components import CombustionChamber
from caldis.components import Turbine
from caldis.solvers import solve  


Initilize paramaters

Ambient Air --> Compressor --> Cooler --> Air Storage --> Combustor --> Turbine --> Exhuast

In [319]:
# --- system definition ---
#Inlet Compressor 
T0=15 +273.15 #k
P0=1.013e5
Eta_ISC=0.85

#Cooler (water)
Tpinch=10 #k


#Air Storage
Pmax=16e5
    #Volume is Variable probably use constraint to find volume based on mass flow rate and time of storage

#Combustion Chamber
Ttit=550 +273.15 #k
Eta_Fuel=0.4

#Turbine
Eta_IST=0.85
Eta_Mech=0.98

Paramters to assume

In [ ]:
# A1: First we define a mass flow rate
Mdot_air=1 #kg/s
Mdot_water=1 #kg/s

# A2: We define a guess for the water outlet temperature
T_water_out_Guess = 300 + 273.15 #K This is an assumption cuz it must be less than 400 and more than T0+Tpinch

# A3: We define the backend for the fluids which is the film convection coefficients
# The heat exchanger is a counter flow heat exchanger with water on one side and air on the other side. 
# The heat transfer coefficients are based on the type of fluid and the flow conditions. Typical values are:
#   forced convection in gas (air): 50–250 W/m²/K
#   forced convection in liquid (water): 500–10000 W/m²/K
h_film_air   = 100.0    # W/m²/K  (air side)
h_film_water = 3000.0   # W/m²/K  (water side)

T_out_comp_guess = 400 + 273.15 #K This is an assumption cuz it must be less than Ttit and more than T0


In [334]:
# --- Component setup ---
be_air   = CoolPropBackend("Air")
be_water = CoolPropBackend("Water")

Source     = FluidBoundary(name="Source", p=P0, T=T0, h=None, mdot=-Mdot_air, backend=be_air)
Comp       = Compressor(name="Comp", backend=be_air, eta_is=Eta_ISC)

# Since Ch<Cc in a counter flow heat exchanger, the pinch point is on the cold side.
# As the outlet of the heat exchanger was set to be T0 + 10, 
# Then the inlet of the water on the cold side would be T0 + 10 - Tpinch= T0
# Pressure is assumed to be atm as it does not affect the heat transfer
Water_Cold = FluidBoundary(name="Water_Cold", p=P0, T=T0, h=None, mdot=-Mdot_water, backend=be_water)
Water_Hot  = FluidBoundary(name="Water_Hot",  p=None, T=None, h=None, mdot=None, backend=be_water)

# The area ratio of the heat exchanger is set to 0.5, 
# meaning that the area of the cold side is half of the area of the hot side. 
# This is a reasonable assumption for a counter flow heat exchanger.
# hfilm defined in A3
Hex = SizingHX(name="Hex", backend_hot=be_air, backend_cold=be_water, h_hot=h_film_air, h_cold=h_film_water, area_ratio_cold_to_hot=0.5)

Sink = FluidBoundary(name="Sink", p=Pmax, T=T0 + Tpinch, h=None, mdot=None, backend=be_air)

In [ ]:
# --- System wiring & start points ---
sys_ = System("Comp_Parts")
sys_.add(Source, Comp, Hex, Sink, Water_Cold, Water_Hot)

sys_.connect(Source.port,     Comp.inl)
sys_.connect(Comp.out,        Hex.hot_in)
sys_.connect(Water_Cold.port, Hex.cold_in)
sys_.connect(Water_Hot.port,  Hex.cold_out)
sys_.connect(Hex.hot_out,     Sink.port)

# Start-point gives the solver a physically sensible initial guess.
# Initially when we calculated the output temp in the 1st step we got 410°C
# The compressor outlet T here isn't known yet, but ~400°C is a good guess 

sys_.set_start_port(Comp.inl,    p=P0,   T=T0,          mdot=Mdot_air)
sys_.set_start_port(Comp.out,    p=Pmax, T=273.15+400,   mdot=-Mdot_air)
sys_.set_start_port(Hex.hot_in,  p=Pmax, T=273.15+400,   mdot=Mdot_air)
sys_.set_start_port(Hex.hot_out, p=Pmax, T=T0+Tpinch,    mdot=-Mdot_air)

# Water side

sys_.set_start_port(Hex.cold_in,  p=P0, T=T0,           mdot=Mdot_water)
sys_.set_start_port(Hex.cold_out, p=P0, T=273.15+100,   mdot=-Mdot_water)

sys_.init_starts()
sol = solve(sys_)

✓ DOF: square (32 unknowns)
✓ start point: evaluable
✓ SOLVED (|F_scaled|=5.0e-10)


In [331]:
#sys_.structure()
Comp.describe()
Hex.describe()

=== Compressor('Comp') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    inl        p=1.013  mdot=+1  h=414.4kJ/kg  T=15.00
    out        p=16.000  mdot=-1  h=822.4kJ/kg  T=410.60
  outputs:
    Wdot             = -408069
    pressure_ratio   = 15.7947
=== SizingHX('Hex') ===
  variables:
    UA               = 4587.68
    A_hot            = 48.9352
  ports (p [bar], mdot [kg/s], T [°C]):
    hot_in     p=16.000  mdot=+1  h=822.4kJ/kg  T=410.60
    hot_out    p=16.000  mdot=-1  h=421.0kJ/kg  T=25.00
    cold_in    p=1.013  mdot=+1  h=63.1kJ/kg  T=15.00
    cold_out   p=1.013  mdot=-1  h=464.5kJ/kg  T=99.97
  outputs:
    U_overall_hot    = 93.75
    U_overall_cold   = 187.5
    A_cold           = 24.4676
    A_total          = 73.4029
    Qdot             = 401395
    DTLM             = 87.4942


{'variables': {'UA': [4587.679369767464], 'A_hot': [48.935246610530726]},
 'ports': {'hot_in': {'p': 1600000.0, 'mdot': 1.0, 'h': 822443.6592563742},
  'hot_out': {'p': 1600000.0, 'mdot': -1.0, 'h': 421048.5357981099},
  'cold_in': {'p': 101300.0, 'mdot': 1.0, 'h': 63076.806869169945},
  'cold_out': {'p': 101300.0, 'mdot': -1.0, 'h': 464471.93040151737}},
 'outputs': {'U_overall_hot': 93.75,
  'U_overall_cold': 187.5,
  'A_cold': 24.467623305265363,
  'A_total': 73.40286991579609,
  'Qdot': 401395.12345826434,
  'DTLM': 87.49415360025472}}